# CAGP NeurIPS 2026 — All Critical Experiments

**Run on Google Colab with GPU runtime (T4 or better)**

This notebook runs all 4 experiments needed for the NeurIPS submission:

| # | Experiment | Purpose | Est. Time (T4) |
|---|---|---|---|
| 1 | Margin Loss Ablation | w_unc=0.1 vs 0.0 → rules out training signal asymmetry | ~20 min |
| 2 | Baseline+Coverage | Energy+Cov, MCDropout+Cov → validates CAGP's novelty | ~25 min |
| 3 | GDELT Pipeline | New temporal KG → addresses "only ICEWS14" critique | ~40 min |
| 4 | R-GCN / CompGCN | GNN baselines → addresses "outdated baselines" critique | ~30 min |

All experiments: 30 epochs, 3 seeds (42, 123, 456), WN18RR + FB15k-237 (+ GDELT for #3).

Results are saved as JSON files that can be downloaded and copied into `outputs/`.

In [ ]:
# Cell 1: Setup
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
import json, gc, time, os, random, urllib.request
from pathlib import Path
from collections import defaultdict

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

os.makedirs('outputs', exist_ok=True)
os.makedirs('data', exist_ok=True)

In [ ]:
# Cell 2: Configuration
EPOCHS = 30
DIM = 100
BATCH_SIZE = 1024  # Matches paper Appendix C.2
LR = 0.001
KL_WEIGHT = 0.001
NUM_BASES = 10  # R-GCN basis decomposition

# Per-dataset seed counts (paper: "Base seeds {42,123,456}; extended to 5 for WN18RR/ICEWS14, 10 for FB15k-237")
SEEDS_3 = [42, 123, 456]
SEEDS_5 = [42, 123, 456, 789, 1011]
SEEDS_10 = [42, 123, 456, 789, 1011, 1213, 1415, 1617, 1819, 2021]

DATASET_SEEDS = {
    'WN18RR': SEEDS_5,
    'FB15k-237': SEEDS_10,
    'YAGO3-10': SEEDS_3,  # Large dataset — 3 seeds for tractability
    'ICEWS14': SEEDS_5,
    'ICEWS18': SEEDS_5,
    'GDELT': SEEDS_3,
}

# Backward compat
SEEDS = SEEDS_3

print(f'Config: epochs={EPOCHS}, dim={DIM}, batch={BATCH_SIZE}')
print(f'Seeds per dataset: WN18RR=5, FB15k-237=10, YAGO3-10=3, ICEWS14=5, ICEWS18=5, GDELT=3')

## Data Loading

Downloads WN18RR, FB15k-237, and GDELT from standard sources.

In [ ]:
# Cell 3: Data loaders
import os
import urllib.request
import gzip
import shutil

def download_dataset(name, url_base):
    dst_dir = f'data/{name}'
    os.makedirs(dst_dir, exist_ok=True)
    
    if name == 'WN18RR':
        splits = ['train.txt', 'valid.txt', 'test.txt']
    elif name == 'FB15k-237':
        splits = ['train.txt', 'valid.txt', 'test.txt']
    else:
        splits = ['train.txt', 'valid.txt', 'test.txt']
    
    for split in splits:
        path = f'{dst_dir}/{split}'
        if not os.path.exists(path):
            url = f'{url_base}/{split}'
            try:
                print(f'  Downloading {name}/{split}...')
                urllib.request.urlretrieve(url, path)
            except urllib.error.HTTPError as e:
                print(f'  Skipping {split} ({e})')
    
    return dst_dir

def load_triples(data_dir):
    '''Load static KG triples. Returns dict of tensors.'''
    import torch
    raw_triples = {'train': [], 'valid': [], 'test': []}
    ent_idx, rel_idx = {}, {}
    next_ent, next_rel = 0, 0
    
    for split in ['train', 'valid', 'test']:
        path = f'{data_dir}/{split}.txt'
        if not os.path.exists(path):
            print(f'  WARNING: {path} not found, skipping')
            continue
        with open(path) as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) < 3:
                    continue
                h, r, t = parts[0].strip(), parts[1].strip(), parts[2].strip()
                if h not in ent_idx:
                    ent_idx[h] = next_ent
                    next_ent += 1
                if t not in ent_idx:
                    ent_idx[t] = next_ent
                    next_ent += 1
                if r not in rel_idx:
                    rel_idx[r] = next_rel
                    next_rel += 1
                raw_triples[split].append((ent_idx[h], rel_idx[r], ent_idx[t]))
    
    tensors = {}
    for split, trips in raw_triples.items():
        if trips:
            tensors[split] = torch.tensor(trips, dtype=torch.long)
        else:
            tensors[split] = torch.zeros(0, 3, dtype=torch.long)
    
    return tensors, next_ent, next_rel

def load_triples_temporal(data_dir):
    '''Load temporal KG triples with timestamps. Returns dict of tensors.'''
    import torch
    raw_triples = {'train': [], 'valid': [], 'test': []}
    raw_ts = {'train': [], 'valid': [], 'test': []}
    ent_idx, rel_idx = {}, {}
    next_ent, next_rel = 0, 0
    
    for split in ['train', 'valid', 'test']:
        path = f'{data_dir}/{split}.txt'
        if not os.path.exists(path):
            print(f'  WARNING: {path} not found, skipping')
            continue
        with open(path) as f:
            for line in f:
                parts = line.strip().split('\t')
                if len(parts) < 3:
                    continue
                h, r, t = parts[0].strip(), parts[1].strip(), parts[2].strip()
                ts = int(parts[3].strip()) if len(parts) >= 4 else 0
                
                if h not in ent_idx:
                    ent_idx[h] = next_ent
                    next_ent += 1
                if t not in ent_idx:
                    ent_idx[t] = next_ent
                    next_ent += 1
                if r not in rel_idx:
                    rel_idx[r] = next_rel
                    next_rel += 1
                
                raw_triples[split].append((ent_idx[h], rel_idx[r], ent_idx[t]))
                raw_ts[split].append(ts)
    
    tensors = {}
    ts_tensors = {}
    for split in raw_triples:
        if raw_triples[split]:
            tensors[split] = torch.tensor(raw_triples[split], dtype=torch.long)
            ts_tensors[split] = torch.tensor(raw_ts[split], dtype=torch.long)
        else:
            tensors[split] = torch.zeros(0, 3, dtype=torch.long)
            ts_tensors[split] = torch.zeros(0, dtype=torch.long)
    
    return tensors, ts_tensors, next_ent, next_rel

# Download WN18RR
print('Downloading WN18RR...')
wn_dir = download_dataset('WN18RR', 
    'https://raw.githubusercontent.com/DeepGraphLearning/KnowledgeGraphEmbedding/master/data/wn18rr')
wn_data, wn_nent, wn_nrel = load_triples(wn_dir)
if 'train' not in wn_data:
    raise RuntimeError(f'WN18RR download failed — check URL and try re-running this cell')
print(f'  WN18RR: {wn_nent} entities, {wn_nrel} relations, {len(wn_data["train"])} train, {len(wn_data["test"])} test')

# Download FB15k-237
print('Downloading FB15k-237...')
fb_dir = download_dataset('FB15k-237',
    'https://raw.githubusercontent.com/DeepGraphLearning/KnowledgeGraphEmbedding/master/data/FB15k-237')
fb_data, fb_nent, fb_nrel = load_triples(fb_dir)
print(f'  FB15k-237: {fb_nent} entities, {fb_nrel} relations, {len(fb_data["train"])} train, {len(fb_data["test"])} test')

# Download ICEWS14
print('Downloading ICEWS14...')
ic14_dir = 'data/ICEWS14'
os.makedirs(ic14_dir, exist_ok=True)
ic14_base = 'https://raw.githubusercontent.com/Liyyy2122/TiRGN/main/data/ICEWS14'
for split in ['train', 'valid', 'test']:
    path = f'{ic14_dir}/{split}.txt'
    if not os.path.exists(path):
        try:
            print(f'  Downloading ICEWS14/{split}.txt...')
            urllib.request.urlretrieve(f'{ic14_base}/{split}.txt', path)
        except urllib.error.HTTPError as e:
            print(f'  Skipping {split}.txt ({e})')
ic14_data, ic14_timestamps, ic14_nent, ic14_nrel = load_triples_temporal(ic14_dir)
print(f'  ICEWS14: {ic14_nent} entities, {ic14_nrel} relations, {len(ic14_data["train"])} train, {len(ic14_data["test"])} test')

# Download GDELT (guarded with try/except)
gd_data = None
try:
    print('Downloading GDELT...')
    gd_dir = 'data/GDELT'
    os.makedirs(gd_dir, exist_ok=True)
    gd_base = 'https://raw.githubusercontent.com/INK-USC/RE-Net/master/data/GDELT'
    for split in ['train', 'valid', 'test']:
        path = f'{gd_dir}/{split}.txt'
        if not os.path.exists(path):
            try:
                print(f'  Downloading GDELT/{split}.txt...')
                urllib.request.urlretrieve(f'{gd_base}/{split}.txt', path)
            except urllib.error.HTTPError as e:
                print(f'  Skipping {split}.txt ({e})')
    gd_data, gd_timestamps, gd_nent, gd_nrel = load_triples_temporal(gd_dir)
    print(f'  GDELT: {gd_nent} entities, {gd_nrel} relations, {len(gd_data["train"])} train, {len(gd_data["test"])} test')
except Exception as e:
    print(f'  WARNING: GDELT download failed ({e}) — will skip GDELT experiments')
    gd_data = None

# Download YAGO3-10
print('Downloading YAGO3-10...')
yago_dir = download_dataset('YAGO3-10',
    'https://raw.githubusercontent.com/DeepGraphLearning/KnowledgeGraphEmbedding/master/data/YAGO3-10')
yago_data, yago_nent, yago_nrel = load_triples(yago_dir)
print(f'  YAGO3-10: {yago_nent} entities, {yago_nrel} relations, {len(yago_data["train"])} train, {len(yago_data["test"])} test')

# Download ICEWS18
print('Downloading ICEWS18...')
icews18_dir = 'data/ICEWS18'
os.makedirs(icews18_dir, exist_ok=True)
icews18_base = 'https://raw.githubusercontent.com/Liyyy2122/TiRGN/main/data/ICEWS18'
for split in ['train', 'valid', 'test']:
    path = f'{icews18_dir}/{split}.txt'
    if not os.path.exists(path):
        try:
            print(f'  Downloading ICEWS18/{split}.txt...')
            urllib.request.urlretrieve(f'{icews18_base}/{split}.txt', path)
        except urllib.error.HTTPError as e:
            print(f'  Skipping {split}.txt ({e})')
ic18_data, ic18_timestamps, ic18_nent, ic18_nrel = load_triples_temporal(icews18_dir)
print(f'  ICEWS18: {ic18_nent} entities, {ic18_nrel} relations, {len(ic18_data["train"])} train, {len(ic18_data["test"])} test')

# Collect datasets
DATASETS = {
    'WN18RR': (wn_data, wn_nent, wn_nrel),
    'FB15k-237': (fb_data, fb_nent, fb_nrel),
    'YAGO3-10': (yago_data, yago_nent, yago_nrel),
}
if gd_data is not None:
    DATASETS_WITH_GDELT = {**DATASETS, 'GDELT': (gd_data, gd_nent, gd_nrel)}
else:
    DATASETS_WITH_GDELT = DATASETS
    print('WARNING: GDELT download failed — skipping GDELT experiments')
print('\nAll datasets loaded!')

## Model Definitions

All models are self-contained — no repo imports needed.

In [ ]:
# Cell 4: Core models (CAGP, GPOnly, CoverageOnly, Energy, MCDropout)
# Bug fixes applied: #1 (MCDropout state), #3 (CAGP normalization), #4 (KL normalization)

class CAGP(nn.Module):
    """Coverage-Augmented GP-KGE."""
    def __init__(self, num_entities, num_relations, dim=DIM):
        super().__init__()
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.entity_mean = nn.Parameter(torch.randn(num_entities, dim) * 0.1)
        self.entity_logvar = nn.Parameter(torch.zeros(num_entities, dim) - 1.0)
        self.relation_emb = nn.Embedding(num_relations, dim)
        self.register_buffer('coverage', torch.zeros(num_entities, num_relations))
        self.alpha_logit = nn.Parameter(torch.tensor(0.0))
        self._norm_stats = None  # Bug #3 fix: cached normalization

    def forward(self, h, r, t):
        if self.training:
            h_std = torch.exp(0.5 * self.entity_logvar[h])
            t_std = torch.exp(0.5 * self.entity_logvar[t])
            h_emb = self.entity_mean[h] + h_std * torch.randn_like(h_std)
            t_emb = self.entity_mean[t] + t_std * torch.randn_like(t_std)
        else:
            h_emb = self.entity_mean[h]
            t_emb = self.entity_mean[t]
        return (h_emb * self.relation_emb(r) * t_emb).sum(-1)

    def get_uncertainty(self, h, r, t):
        h_var = torch.exp(self.entity_logvar[h]).mean(dim=-1)
        t_var = torch.exp(self.entity_logvar[t]).mean(dim=-1)
        gp_var = (h_var + t_var) / 2
        cov_unc = 2.0 - self.coverage[h, r] - self.coverage[t, r]
        # Bug #3 fix: use cached stats for reproducible normalization
        if self._norm_stats is not None:
            gp_mean = self._norm_stats['gp_mean']
            cov_mean = self._norm_stats['cov_mean']
        else:
            gp_mean = gp_var.mean().item()
            cov_mean = cov_unc.mean().item()
        gp_norm = gp_var / (gp_mean + 1e-8) * (cov_mean + 1e-8)
        alpha = torch.sigmoid(self.alpha_logit)
        return alpha * gp_norm + (1 - alpha) * cov_unc

    def precompute_coverage(self, triples):
        for i in range(len(triples)):
            self.coverage[triples[i, 0], triples[i, 1]] = 1.0
            self.coverage[triples[i, 2], triples[i, 1]] = 1.0

    def kl_loss(self):
        # Bug #4 fix: normalize by num_entities, not num_elements
        kl = -0.5 * torch.sum(1 + self.entity_logvar - self.entity_mean.pow(2) - self.entity_logvar.exp())
        return kl / self.num_entities


class GPOnly(nn.Module):
    """GP variance only (semantic uncertainty)."""
    def __init__(self, num_entities, num_relations, dim=DIM):
        super().__init__()
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.entity_mean = nn.Parameter(torch.randn(num_entities, dim) * 0.1)
        self.entity_logvar = nn.Parameter(torch.zeros(num_entities, dim) - 1.0)
        self.relation_emb = nn.Embedding(num_relations, dim)
        self.register_buffer('coverage', torch.zeros(num_entities, num_relations))

    def forward(self, h, r, t):
        if self.training:
            h_std = torch.exp(0.5 * self.entity_logvar[h])
            t_std = torch.exp(0.5 * self.entity_logvar[t])
            h_emb = self.entity_mean[h] + h_std * torch.randn_like(h_std)
            t_emb = self.entity_mean[t] + t_std * torch.randn_like(t_std)
        else:
            h_emb = self.entity_mean[h]
            t_emb = self.entity_mean[t]
        return (h_emb * self.relation_emb(r) * t_emb).sum(-1)

    def get_uncertainty(self, h, r, t):
        h_var = torch.exp(self.entity_logvar[h]).mean(dim=-1)
        t_var = torch.exp(self.entity_logvar[t]).mean(dim=-1)
        return (h_var + t_var) / 2

    def precompute_coverage(self, triples):
        for i in range(len(triples)):
            self.coverage[triples[i, 0], triples[i, 1]] = 1.0
            self.coverage[triples[i, 2], triples[i, 1]] = 1.0

    def kl_loss(self):
        # Bug #4 fix: normalize by num_entities
        kl = -0.5 * torch.sum(1 + self.entity_logvar - self.entity_mean.pow(2) - self.entity_logvar.exp())
        return kl / self.num_entities


class CoverageOnly(nn.Module):
    """Coverage-only uncertainty."""
    def __init__(self, num_entities, num_relations, dim=DIM):
        super().__init__()
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.entity_emb = nn.Embedding(num_entities, dim)
        self.relation_emb = nn.Embedding(num_relations, dim)
        self.register_buffer('coverage', torch.zeros(num_entities, num_relations))

    def forward(self, h, r, t):
        return (self.entity_emb(h) * self.relation_emb(r) * self.entity_emb(t)).sum(-1)

    def get_uncertainty(self, h, r, t):
        return 2.0 - self.coverage[h, r] - self.coverage[t, r]

    def precompute_coverage(self, triples):
        for i in range(len(triples)):
            self.coverage[triples[i, 0], triples[i, 1]] = 1.0
            self.coverage[triples[i, 2], triples[i, 1]] = 1.0

    def kl_loss(self):
        return torch.tensor(0.0)


class EnergyBasedKGE(nn.Module):
    """DistMult + Energy-based uncertainty."""
    def __init__(self, num_entities, num_relations, dim=DIM):
        super().__init__()
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.entity_emb = nn.Embedding(num_entities, dim)
        self.relation_emb = nn.Embedding(num_relations, dim)
        self.register_buffer('coverage', torch.zeros(num_entities, num_relations))

    def forward(self, h, r, t):
        return (self.entity_emb(h) * self.relation_emb(r) * self.entity_emb(t)).sum(-1)

    def get_uncertainty(self, h, r, t):
        return -self.forward(h, r, t)

    def precompute_coverage(self, triples):
        for i in range(len(triples)):
            self.coverage[triples[i, 0], triples[i, 1]] = 1.0
            self.coverage[triples[i, 2], triples[i, 1]] = 1.0

    def kl_loss(self):
        return torch.tensor(0.0)


class MCDropoutKGE(nn.Module):
    """DistMult + MC Dropout for uncertainty."""
    def __init__(self, num_entities, num_relations, dim=DIM, dropout=0.1, num_samples=10):
        super().__init__()
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.num_samples = num_samples
        self.entity_emb = nn.Embedding(num_entities, dim)
        self.relation_emb = nn.Embedding(num_relations, dim)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer('coverage', torch.zeros(num_entities, num_relations))

    def forward(self, h, r, t, use_dropout=False):
        # Bug #1 fix: explicit dropout control instead of relying on model state
        h_e = self.entity_emb(h)
        t_e = self.entity_emb(t)
        if use_dropout or self.training:
            h_e = self.dropout(h_e)
            t_e = self.dropout(t_e)
        return (h_e * self.relation_emb(r) * t_e).sum(-1)

    def get_uncertainty(self, h, r, t):
        # Bug #1 fix: save and restore training state
        was_training = self.training
        self.train()  # Enable dropout
        scores = torch.stack([self.forward(h, r, t, use_dropout=True) for _ in range(self.num_samples)])
        if not was_training:
            self.eval()
        return scores.var(dim=0)

    def precompute_coverage(self, triples):
        for i in range(len(triples)):
            self.coverage[triples[i, 0], triples[i, 1]] = 1.0
            self.coverage[triples[i, 2], triples[i, 1]] = 1.0

    def kl_loss(self):
        return torch.tensor(0.0)


print('All core models defined.')

class RelCondVar(nn.Module):
    """Relation-Conditioned Variance — learns per-relation variance parameters."""
    def __init__(self, num_entities, num_relations, dim=DIM):
        super().__init__()
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.entity_mean = nn.Parameter(torch.randn(num_entities, dim) * 0.1)
        self.entity_logvar = nn.Parameter(torch.zeros(num_entities, dim) - 1.0)
        self.relation_emb = nn.Embedding(num_relations, dim)
        # Per-relation variance adjustment (initialized to high uncertainty)
        self.rel_logvar_offset = nn.Parameter(torch.zeros(num_relations, dim))
        self.register_buffer('coverage', torch.zeros(num_entities, num_relations))

    def forward(self, h, r, t):
        if self.training:
            h_logvar = self.entity_logvar[h] + self.rel_logvar_offset[r]
            t_logvar = self.entity_logvar[t] + self.rel_logvar_offset[r]
            h_std = torch.exp(0.5 * h_logvar)
            t_std = torch.exp(0.5 * t_logvar)
            h_emb = self.entity_mean[h] + h_std * torch.randn_like(h_std)
            t_emb = self.entity_mean[t] + t_std * torch.randn_like(t_std)
        else:
            h_emb = self.entity_mean[h]
            t_emb = self.entity_mean[t]
        return (h_emb * self.relation_emb(r) * t_emb).sum(-1)

    def get_uncertainty(self, h, r, t):
        h_logvar = self.entity_logvar[h] + self.rel_logvar_offset[r]
        t_logvar = self.entity_logvar[t] + self.rel_logvar_offset[r]
        h_var = torch.exp(h_logvar).mean(dim=-1)
        t_var = torch.exp(t_logvar).mean(dim=-1)
        return (h_var + t_var) / 2

    def precompute_coverage(self, triples):
        pass  # Uses vectorized version from train_model

    def kl_loss(self):
        kl = -0.5 * torch.sum(1 + self.entity_logvar - self.entity_mean.pow(2) - self.entity_logvar.exp())
        return kl / self.num_entities


class CAGPNoReparam(nn.Module):
    """CAGP WITHOUT reparameterization sampling — for ablation."""
    def __init__(self, num_entities, num_relations, dim=DIM):
        super().__init__()
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.entity_mean = nn.Parameter(torch.randn(num_entities, dim) * 0.1)
        self.entity_logvar = nn.Parameter(torch.zeros(num_entities, dim) - 1.0)
        self.relation_emb = nn.Embedding(num_relations, dim)
        self.register_buffer('coverage', torch.zeros(num_entities, num_relations))
        self.alpha_logit = nn.Parameter(torch.tensor(0.0))
        self._norm_stats = None

    def forward(self, h, r, t):
        # NO reparameterization — always use mean (causes KL-driven variance collapse)
        h_emb = self.entity_mean[h]
        t_emb = self.entity_mean[t]
        return (h_emb * self.relation_emb(r) * t_emb).sum(-1)

    def get_uncertainty(self, h, r, t):
        h_var = torch.exp(self.entity_logvar[h]).mean(dim=-1)
        t_var = torch.exp(self.entity_logvar[t]).mean(dim=-1)
        gp_var = (h_var + t_var) / 2
        cov_unc = 2.0 - self.coverage[h, r] - self.coverage[t, r]
        if self._norm_stats is not None:
            gp_mean = self._norm_stats['gp_mean']
            cov_mean = self._norm_stats['cov_mean']
        else:
            gp_mean = gp_var.mean().item()
            cov_mean = cov_unc.mean().item()
        gp_norm = gp_var / (gp_mean + 1e-8) * (cov_mean + 1e-8)
        alpha = torch.sigmoid(self.alpha_logit)
        return alpha * gp_norm + (1 - alpha) * cov_unc

    def precompute_coverage(self, triples):
        pass

    def kl_loss(self):
        kl = -0.5 * torch.sum(1 + self.entity_logvar - self.entity_mean.pow(2) - self.entity_logvar.exp())
        return kl / self.num_entities


class UKGE(nn.Module):
    """UKGE-style confidence scoring (simplified)."""
    def __init__(self, num_entities, num_relations, dim=DIM):
        super().__init__()
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.entity_emb = nn.Embedding(num_entities, dim)
        self.relation_emb = nn.Embedding(num_relations, dim)
        self.confidence_head = nn.Sequential(nn.Linear(dim * 3, dim), nn.ReLU(), nn.Linear(dim, 1))
        self.register_buffer('coverage', torch.zeros(num_entities, num_relations))

    def forward(self, h, r, t):
        return (self.entity_emb(h) * self.relation_emb(r) * self.entity_emb(t)).sum(-1)

    def get_uncertainty(self, h, r, t):
        feat = torch.cat([self.entity_emb(h), self.relation_emb(r), self.entity_emb(t)], dim=-1)
        conf = torch.sigmoid(self.confidence_head(feat).squeeze(-1))
        return 1.0 - conf

    def precompute_coverage(self, triples):
        pass

    def kl_loss(self):
        return torch.tensor(0.0)


class SNGPModel(nn.Module):
    """Spectral-Normalized GP output layer for KGE."""
    def __init__(self, num_entities, num_relations, dim=DIM):
        super().__init__()
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.entity_emb = nn.Embedding(num_entities, dim)
        self.relation_emb = nn.Embedding(num_relations, dim)
        # GP approximation: random Fourier features
        self.n_features = 128
        self.rff_weight = nn.Parameter(torch.randn(dim, self.n_features) * 0.1, requires_grad=False)
        self.rff_bias = nn.Parameter(torch.rand(self.n_features) * 2 * 3.14159, requires_grad=False)
        self.beta = nn.Parameter(torch.zeros(self.n_features))
        self.register_buffer('precision', torch.eye(self.n_features) * 1.0)
        self.register_buffer('coverage', torch.zeros(num_entities, num_relations))

    def _phi(self, x):
        return (2.0 / self.n_features) ** 0.5 * torch.cos(x @ self.rff_weight + self.rff_bias)

    def forward(self, h, r, t):
        return (self.entity_emb(h) * self.relation_emb(r) * self.entity_emb(t)).sum(-1)

    def get_uncertainty(self, h, r, t):
        feat = (self.entity_emb(h) + self.entity_emb(t)) / 2
        phi = self._phi(feat)
        # GP predictive variance: phi^T @ precision^{-1} @ phi
        var = (phi @ torch.inverse(self.precision) * phi).sum(-1)
        return var

    def precompute_coverage(self, triples):
        pass

    def kl_loss(self):
        return torch.tensor(0.0)


class DeepEnsembleKGE(nn.Module):
    """Wrapper for Deep Ensembles — trains N independent models."""
    def __init__(self, num_entities, num_relations, dim=DIM, n_members=3):
        super().__init__()
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.n_members = n_members
        self.members = nn.ModuleList([
            self._make_member(num_entities, num_relations, dim) for _ in range(n_members)
        ])
        self.register_buffer('coverage', torch.zeros(num_entities, num_relations))

    def _make_member(self, ne, nr, dim):
        m = nn.Module()
        m.entity_emb = nn.Embedding(ne, dim)
        m.relation_emb = nn.Embedding(nr, dim)
        m.forward = lambda h, r, t, _m=m: (_m.entity_emb(h) * _m.relation_emb(r) * _m.entity_emb(t)).sum(-1)
        return m

    def forward(self, h, r, t):
        scores = torch.stack([m.forward(h, r, t) for m in self.members])
        return scores.mean(0)

    def get_uncertainty(self, h, r, t):
        scores = torch.stack([m.forward(h, r, t) for m in self.members])
        return scores.var(0)

    def precompute_coverage(self, triples):
        pass

    def kl_loss(self):
        return torch.tensor(0.0)

In [ ]:
# Cell 5: R-GCN and CompGCN models

class RGCNLayer(nn.Module):
    """Relational Graph Convolution with basis decomposition."""
    def __init__(self, in_dim, out_dim, num_relations, num_bases=NUM_BASES):
        super().__init__()
        self.num_bases = min(num_bases, num_relations)
        self.bases = nn.Parameter(torch.randn(self.num_bases, in_dim, out_dim) * 0.01)
        self.coefficients = nn.Parameter(torch.randn(num_relations, self.num_bases) * 0.01)
        self.self_loop = nn.Linear(in_dim, out_dim, bias=False)
        self.bias = nn.Parameter(torch.zeros(out_dim))

    def forward(self, entity_emb, edge_index, edge_type):
        num_entities = entity_emb.size(0)
        out = torch.zeros(num_entities, self.bias.size(0), device=entity_emb.device)
        W = torch.einsum('rb,bio->rio', self.coefficients, self.bases)
        src, dst = edge_index[0], edge_index[1]
        for r in range(W.size(0)):
            mask = edge_type == r
            if mask.sum() == 0:
                continue
            msg = entity_emb[src[mask]] @ W[r]
            out.index_add_(0, dst[mask], msg)
        in_degree = torch.zeros(num_entities, device=entity_emb.device)
        in_degree.index_add_(0, dst, torch.ones(len(dst), device=entity_emb.device))
        out = out / in_degree.clamp(min=1).unsqueeze(1)
        return F.relu(out + self.self_loop(entity_emb) + self.bias)


class RGCNEncoder(nn.Module):
    def __init__(self, num_entities, num_relations, dim=DIM):
        super().__init__()
        self.entity_emb = nn.Embedding(num_entities, dim)
        nn.init.xavier_uniform_(self.entity_emb.weight)
        self.layer1 = RGCNLayer(dim, dim, num_relations * 2)
        self.layer2 = RGCNLayer(dim, dim, num_relations * 2)

    def forward(self, edge_index, edge_type):
        x = self.entity_emb.weight
        x = self.layer1(x, edge_index, edge_type)
        return self.layer2(x, edge_index, edge_type)


class CompGCNLayer(nn.Module):
    def __init__(self, in_dim, out_dim, composition='sub'):
        super().__init__()
        self.composition = composition
        self.W_in = nn.Linear(in_dim, out_dim, bias=False)
        self.W_out = nn.Linear(in_dim, out_dim, bias=False)
        self.W_self = nn.Linear(in_dim, out_dim, bias=False)
        self.W_rel = nn.Linear(in_dim, out_dim, bias=False)
        self.bias = nn.Parameter(torch.zeros(out_dim))
        self.bn = nn.BatchNorm1d(out_dim)

    def compose(self, ent, rel):
        if self.composition == 'sub':
            return ent - rel
        elif self.composition == 'mult':
            return ent * rel
        else:
            return torch.fft.irfft(torch.fft.rfft(ent) * torch.conj(torch.fft.rfft(rel)), n=ent.size(-1))

    def forward(self, entity_emb, relation_emb, edge_index, edge_type, n_rel_orig):
        n_ent = entity_emb.size(0)
        out = torch.zeros(n_ent, self.W_in.out_features, device=entity_emb.device)
        src, dst = edge_index[0], edge_index[1]
        fwd = edge_type < n_rel_orig
        if fwd.sum() > 0:
            msg = self.W_in(self.compose(entity_emb[src[fwd]], relation_emb[edge_type[fwd]]))
            out.index_add_(0, dst[fwd], msg)
        inv = edge_type >= n_rel_orig
        if inv.sum() > 0:
            msg = self.W_out(self.compose(entity_emb[src[inv]], relation_emb[edge_type[inv] - n_rel_orig]))
            out.index_add_(0, dst[inv], msg)
        deg = torch.zeros(n_ent, device=entity_emb.device)
        deg.index_add_(0, dst, torch.ones(len(dst), device=entity_emb.device))
        out = out / deg.clamp(min=1).unsqueeze(1)
        out = self.bn(F.relu(out + self.W_self(entity_emb) + self.bias))
        return out, self.W_rel(relation_emb)


class CompGCNEncoder(nn.Module):
    def __init__(self, num_entities, num_relations, dim=DIM):
        super().__init__()
        self.num_relations = num_relations
        self.entity_emb = nn.Embedding(num_entities, dim)
        self.relation_emb = nn.Embedding(num_relations, dim)
        nn.init.xavier_uniform_(self.entity_emb.weight)
        nn.init.xavier_uniform_(self.relation_emb.weight)
        self.layer1 = CompGCNLayer(dim, dim)
        self.layer2 = CompGCNLayer(dim, dim)

    def forward(self, edge_index, edge_type):
        x, r = self.entity_emb.weight, self.relation_emb.weight
        x, r = self.layer1(x, r, edge_index, edge_type, self.num_relations)
        x, r = self.layer2(x, r, edge_index, edge_type, self.num_relations)
        return x, r


class GNNModel(nn.Module):
    """GNN encoder + DistMult scoring + optional coverage."""
    def __init__(self, num_entities, num_relations, dim=DIM, encoder_type='rgcn', use_coverage=False):
        super().__init__()
        self.num_entities = num_entities
        self.num_relations = num_relations
        self.encoder_type = encoder_type
        self.use_coverage = use_coverage
        if encoder_type == 'rgcn':
            self.encoder = RGCNEncoder(num_entities, num_relations, dim)
            self.relation_emb = nn.Embedding(num_relations, dim)
        else:
            self.encoder = CompGCNEncoder(num_entities, num_relations, dim)
        self.register_buffer('coverage', torch.zeros(num_entities, num_relations))
        self.register_buffer('edge_index', torch.zeros(2, 0, dtype=torch.long))
        self.register_buffer('edge_type', torch.zeros(0, dtype=torch.long))
        self._entity_embeddings = None

    def build_graph(self, triples):
        h, r, t = triples[:, 0], triples[:, 1], triples[:, 2]
        self.edge_index = torch.stack([torch.cat([h, t]), torch.cat([t, h])])
        self.edge_type = torch.cat([r, r + self.num_relations])

    def encode(self):
        if self.encoder_type == 'rgcn':
            self._entity_embeddings = self.encoder(self.edge_index, self.edge_type)
        else:
            self._entity_embeddings, self._rel_embeddings = self.encoder(self.edge_index, self.edge_type)

    def forward(self, h, r, t):
        if self._entity_embeddings is None:
            self.encode()
        h_e, t_e = self._entity_embeddings[h], self._entity_embeddings[t]
        r_e = self._rel_embeddings[r] if (self.encoder_type == 'compgcn' and hasattr(self, '_rel_embeddings')) else self.relation_emb(r)
        return (h_e * r_e * t_e).sum(-1)

    def get_uncertainty(self, h, r, t):
        energy_unc = -self.forward(h, r, t)
        if self.use_coverage:
            struct = 2.0 - self.coverage[h, r] - self.coverage[t, r]
            e_min, e_max = energy_unc.min(), energy_unc.max()
            energy_norm = 2.0 * (energy_unc - e_min) / (e_max - e_min + 1e-8)
            return 0.5 * energy_norm + 0.5 * struct
        return energy_unc

    def precompute_coverage(self, triples):
        for i in range(len(triples)):
            h, r, t = triples[i, 0].item(), triples[i, 1].item(), triples[i, 2].item()
            if h < self.num_entities and r < self.num_relations:
                self.coverage[h, r] = 1
            if t < self.num_entities and r < self.num_relations:
                self.coverage[t, r] = 1

    def kl_loss(self):
        return torch.tensor(0.0)


print('GNN models defined.')

In [ ]:
# Cell 6: Shared training and evaluation functions
# Bug #3 fix: calibration helper for CAGP normalization
# Bug #2 REVERTED: per-batch GNN encoding is necessary for correct gradient flow
# Bug #5 REVERTED: per-sample margin loss is standard KGE practice
# Device fixes: all indexing operations handle GPU model + CPU triples correctly
# Perf fix: vectorized precompute_coverage (critical for GDELT ~2M triples)

def calibrate_normalization(model, triples):
    """Bug #3 fix: cache normalization stats from training data for reproducible eval."""
    if not isinstance(model, CAGP):
        return
    sample_size = min(5000, len(triples))
    idx = torch.randint(0, len(triples), (sample_size,))
    h_s = triples[idx, 0].to(device)
    r_s = triples[idx, 1].to(device)
    t_s = triples[idx, 2].to(device)
    with torch.no_grad():
        h_var = torch.exp(model.entity_logvar[h_s]).mean(dim=-1)
        t_var = torch.exp(model.entity_logvar[t_s]).mean(dim=-1)
        gp_var = (h_var + t_var) / 2
        cov_unc = 2.0 - model.coverage[h_s, r_s] - model.coverage[t_s, r_s]
        model._norm_stats = {
            'gp_mean': gp_var.mean().item(),
            'cov_mean': cov_unc.mean().item(),
        }


def vectorized_precompute_coverage(model, triples):
    """Vectorized coverage precomputation — critical for GDELT (~2M triples).
    The loop-based version takes 30+ minutes; vectorized takes <1 second.
    """
    h_idx = triples[:, 0].to(model.coverage.device)
    r_idx = triples[:, 1].to(model.coverage.device)
    t_idx = triples[:, 2].to(model.coverage.device)
    # Bounds check (needed for GNNModel which may have edge cases)
    valid_h = (h_idx < model.num_entities) & (r_idx < model.num_relations)
    valid_t = (t_idx < model.num_entities) & (r_idx < model.num_relations)
    model.coverage[h_idx[valid_h], r_idx[valid_h]] = 1.0
    model.coverage[t_idx[valid_t], r_idx[valid_t]] = 1.0


def train_model(model, triples, epochs=EPOCHS, unc_weight=0.0, batch_size=BATCH_SIZE, is_gp=False):
    """Train a KGE model. Works for all model types."""
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=LR)

    vectorized_precompute_coverage(model, triples)

    h_all, r_all, t_all = triples[:, 0], triples[:, 1], triples[:, 2]
    dataset = TensorDataset(h_all, r_all, t_all)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                        num_workers=0, pin_memory=False)  # num_workers=0 for Colab compatibility

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for h, r, t in loader:
            h, r, t = h.to(device), r.to(device), t.to(device)
            pos_scores = model(h, r, t)
            neg_t = torch.randint(0, model.num_entities, t.shape, device=device)
            neg_scores = model(h, r, neg_t)
            loss = (F.binary_cross_entropy_with_logits(pos_scores, torch.ones_like(pos_scores)) +
                    F.binary_cross_entropy_with_logits(neg_scores, torch.zeros_like(neg_scores)))
            if is_gp and hasattr(model, 'kl_loss'):
                kl = model.kl_loss()
                if kl.item() != 0:
                    loss += KL_WEIGHT * kl
            if unc_weight > 0 and hasattr(model, 'get_uncertainty'):
                unc_pos = model.get_uncertainty(h, r, t)
                unc_neg = model.get_uncertainty(h, r, neg_t)
                # Per-sample hinge margin loss (standard KGE practice)
                margin_loss = F.relu(0.3 - (unc_neg - unc_pos)).mean()
                loss += unc_weight * margin_loss
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
        if (epoch + 1) % 10 == 0:
            print(f'    Epoch {epoch+1}/{epochs}, loss={total_loss/len(loader):.4f}')
    model.eval()
    calibrate_normalization(model, triples)
    return model


def train_gnn_model(model, triples, epochs=EPOCHS, batch_size=BATCH_SIZE):
    """Train GNN-based model.
    NOTE: encode() MUST be called per-batch for correct gradient flow.
    After backward()+step(), the computation graph is consumed, so we must
    re-encode to build a fresh graph for the next batch. On GPU this is fast.
    """
    model = model.to(device)
    model.build_graph(triples)
    vectorized_precompute_coverage(model, triples)
    model.edge_index = model.edge_index.to(device)
    model.edge_type = model.edge_type.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=LR)
    h_all, r_all, t_all = triples[:, 0], triples[:, 1], triples[:, 2]
    dataset = TensorDataset(h_all, r_all, t_all)
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                        num_workers=0, pin_memory=False)  # num_workers=0 for Colab

    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for h, r, t in loader:
            h, r, t = h.to(device), r.to(device), t.to(device)
            model._entity_embeddings = None  # Invalidate before re-encoding
            model.encode()  # Fresh computation graph each batch
            pos_scores = model(h, r, t)
            neg_t = torch.randint(0, model.num_entities, t.shape, device=device)
            neg_scores = model(h, r, neg_t)
            loss = (F.binary_cross_entropy_with_logits(pos_scores, torch.ones_like(pos_scores)) +
                    F.binary_cross_entropy_with_logits(neg_scores, torch.zeros_like(neg_scores)))
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            model._entity_embeddings = None  # Consumed by backward
        if (epoch + 1) % 5 == 0:
            print(f'    Epoch {epoch+1}/{epochs}, loss={total_loss/len(loader):.4f}')
    model.eval()
    model.encode()  # Final encoding for evaluation
    return model


def evaluate_ood(model, train_triples, test_triples):
    """Evaluate OOD detection with AUROC, AUPR, and FPR@95TPR.
    Handles GPU model + CPU triples correctly.
    """
    model.eval()
    # Compute entity frequencies (CPU — no device issues with scalar .item())
    freq = torch.zeros(model.num_entities, dtype=torch.long)
    h_train = train_triples[:, 0]
    t_train = train_triples[:, 2]
    # Vectorized frequency counting (much faster than loop for GDELT)
    freq.scatter_add_(0, h_train, torch.ones_like(h_train))
    freq.scatter_add_(0, t_train, torch.ones_like(t_train))
    tau = int(np.percentile(freq[freq > 0].numpy(), 25))

    h_test = test_triples[:, 0]
    r_test = test_triples[:, 1]
    t_test = test_triples[:, 2]
    min_freq = torch.minimum(freq[h_test], freq[t_test])
    is_emerging = min_freq <= tau

    # Coverage lookup — must move indices to same device as coverage buffer
    cov_h = model.coverage[h_test.to(device), r_test.to(device)].cpu()
    cov_t = model.coverage[t_test.to(device), r_test.to(device)].cpu()
    is_novel = (~is_emerging) & ((cov_h == 0) | (cov_t == 0))
    is_ood = is_emerging | is_novel

    if is_ood.sum() == 0 or (~is_ood).sum() == 0:
        return {'overall': float('nan'), 'emerging': float('nan'), 'novel': float('nan')}

    with torch.no_grad():
        unc = model.get_uncertainty(h_test.to(device), r_test.to(device), t_test.to(device)).cpu().numpy()

    labels = is_ood.numpy().astype(int)
    results = {
        'overall': float(roc_auc_score(labels, unc)),
        'overall_aupr': float(average_precision_score(labels, unc)),
    }
    # FPR@95TPR
    ood_scores = unc[labels == 1]
    id_scores = unc[labels == 0]
    if len(ood_scores) > 0 and len(id_scores) > 0:
        tpr95_threshold = np.percentile(ood_scores, 5)  # 95% of OOD above this
        fpr = (id_scores >= tpr95_threshold).mean()
        results['fpr95'] = float(fpr)

    for name, mask in [('emerging', is_emerging), ('novel', is_novel)]:
        if mask.sum() > 0:
            m = mask | (~is_ood)
            if mask[m].sum() > 0 and (~is_ood)[m].sum() > 0:
                results[name] = float(roc_auc_score(mask[m].numpy(), unc[m]))
    return results


def evaluate_strict_split(model, train_triples, test_triples, num_relations):
    """Non-circular evaluation: remove inverse-relation overlap from test set.
    This is the key evaluation that addresses reviewer concerns about circularity.
    """
    model.eval()
    # Build train set for overlap detection (h,r,t) and (t,r_inv,h)
    train_set = set()
    for i in range(len(train_triples)):
        h, r, t = train_triples[i, 0].item(), train_triples[i, 1].item(), train_triples[i, 2].item()
        train_set.add((h, r, t))
        train_set.add((t, r, h))  # Inverse pattern

    # Filter test triples: remove any that overlap with train (forward or inverse)
    keep = []
    for i in range(len(test_triples)):
        h, r, t = test_triples[i, 0].item(), test_triples[i, 1].item(), test_triples[i, 2].item()
        if (h, r, t) not in train_set and (t, r, h) not in train_set:
            keep.append(i)

    if len(keep) == 0:
        return {'strict_auroc': float('nan'), 'removed_pct': 100.0}

    removed_pct = 100.0 * (1 - len(keep) / len(test_triples))
    strict_test = test_triples[keep]

    # Re-evaluate on filtered test set
    result = evaluate_ood(model, train_triples, strict_test)
    result['removed_pct'] = removed_pct
    result['strict_n_test'] = len(keep)
    return result


def combine_with_coverage(model, test_triples, alpha=0.5):
    """Post-hoc: combine a baseline's uncertainty with coverage signal."""
    h = test_triples[:, 0].to(device)
    r = test_triples[:, 1].to(device)
    t = test_triples[:, 2].to(device)
    with torch.no_grad():
        base_unc = model.get_uncertainty(h, r, t).cpu()
    cov_unc = (2.0 - model.coverage[h, r] - model.coverage[t, r]).cpu()
    base_norm = base_unc / (base_unc.mean() + 1e-8) * (cov_unc.mean() + 1e-8)
    return (alpha * base_norm + (1 - alpha) * cov_unc).numpy()


def set_seed(seed):
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)


print('Training/evaluation functions defined (with strict-split + AUPR + FPR@95).')
print(f'  Device: {device}, num_workers=0 (Colab-safe), vectorized coverage')

---
## Experiment 1: Margin Loss Ablation

Tests CAGP with w_unc=0.1 (default) vs w_unc=0.0 (no margin loss).
Addresses "training signal asymmetry" concern.

In [ ]:
# Cell 7: Experiment 1 — Margin Loss Ablation
print('='*70)
print('EXPERIMENT 1: MARGIN LOSS ABLATION (30 epochs, 3 seeds)')
print('='*70)

exp1_results = {}
t0 = time.time()

for ds_name, (data, n_ent, n_rel) in DATASETS.items():
    print(f'\n--- {ds_name} ({n_ent} ent, {n_rel} rel) ---')
    train_t, test_t = data['train'], data['test']
    seed_results = []

    for seed in DATASET_SEEDS.get(ds_name, SEEDS):
        print(f'\n  Seed {seed}:')
        sr = {}

        # CAGP with margin loss
        print(f'    CAGP w_unc=0.1...')
        set_seed(seed)
        m = CAGP(n_ent, n_rel)
        m = train_model(m, train_t, unc_weight=0.1, is_gp=True)
        sr['cagp_with_margin'] = evaluate_ood(m, train_t, test_t)
        alpha_val = torch.sigmoid(m.alpha_logit).item()
        sr['cagp_with_margin']['alpha'] = alpha_val
        print(f'      AUROC: {sr["cagp_with_margin"]["overall"]:.4f}, alpha={alpha_val:.3f}')
        del m; torch.cuda.empty_cache(); gc.collect()

        # CAGP without margin loss
        print(f'    CAGP w_unc=0.0...')
        set_seed(seed)
        m = CAGP(n_ent, n_rel)
        m = train_model(m, train_t, unc_weight=0.0, is_gp=True)
        sr['cagp_without_margin'] = evaluate_ood(m, train_t, test_t)
        alpha_val = torch.sigmoid(m.alpha_logit).item()
        sr['cagp_without_margin']['alpha'] = alpha_val
        print(f'      AUROC: {sr["cagp_without_margin"]["overall"]:.4f}, alpha={alpha_val:.3f}')
        del m; torch.cuda.empty_cache(); gc.collect()

        # CoverageOnly
        print(f'    CoverageOnly...')
        set_seed(seed)
        m = CoverageOnly(n_ent, n_rel)
        m = train_model(m, train_t)
        sr['coverage_only'] = evaluate_ood(m, train_t, test_t)
        print(f'      AUROC: {sr["coverage_only"]["overall"]:.4f}')
        del m; torch.cuda.empty_cache(); gc.collect()

        # GPOnly
        print(f'    GPOnly...')
        set_seed(seed)
        m = GPOnly(n_ent, n_rel)
        m = train_model(m, train_t, is_gp=True)
        sr['gp_only'] = evaluate_ood(m, train_t, test_t)
        print(f'      AUROC: {sr["gp_only"]["overall"]:.4f}')
        del m; torch.cuda.empty_cache(); gc.collect()

        seed_results.append(sr)

    # Aggregate — include AUPR, FPR@95, alpha
    summary = {}
    for method in seed_results[0]:
        for metric in ['overall', 'overall_aupr', 'fpr95', 'emerging', 'novel', 'alpha']:
            vals = [s[method].get(metric, float('nan')) for s in seed_results]
            vals = [v for v in vals if isinstance(v, (int, float)) and not np.isnan(v)]
            if vals:
                summary[f'{method}_{metric}'] = {'mean': float(np.mean(vals)), 'std': float(np.std(vals))}
    exp1_results[ds_name] = {'per_seed': seed_results, 'summary': summary}

exp1_results['elapsed_seconds'] = time.time() - t0
with open('outputs/exp1_margin_loss_ablation.json', 'w') as f:
    json.dump(exp1_results, f, indent=2)

print(f'\n\nExp 1 done in {exp1_results["elapsed_seconds"]:.0f}s')
for ds in DATASETS:
    print(f'\n{ds}:')
    for k, v in sorted(exp1_results[ds]['summary'].items()):
        print(f'  {k}: {v["mean"]:.4f} ± {v["std"]:.4f}')
    # Key comparison
    s = exp1_results[ds]['summary']
    with_m = s.get('cagp_with_margin_overall', {}).get('mean', 0)
    without_m = s.get('cagp_without_margin_overall', {}).get('mean', 0)
    print(f'\n  Margin loss contribution: {with_m - without_m:+.4f}')
    alpha_with = s.get('cagp_with_margin_alpha', {}).get('mean', 0)
    alpha_without = s.get('cagp_without_margin_alpha', {}).get('mean', 0)
    print(f'  Learned alpha (w/ margin): {alpha_with:.3f}, (w/o): {alpha_without:.3f}')


---
## Experiment 2: Baseline + Coverage Ablation

Tests: Energy+Cov, MCDropout+Cov vs CAGP.
Shows that naively adding coverage to any baseline does NOT match CAGP.

In [ ]:
# Cell 8: Experiment 2 — Baseline + Coverage
print('='*70)
print('EXPERIMENT 2: BASELINE + COVERAGE ABLATION (30 epochs, 3 seeds)')
print('='*70)

exp2_results = {}
t0 = time.time()

for ds_name, (data, n_ent, n_rel) in DATASETS.items():
    print(f'\n--- {ds_name} ---')
    train_t, test_t = data['train'], data['test']
    seed_results = []

    for seed in DATASET_SEEDS.get(ds_name, SEEDS):
        print(f'\n  Seed {seed}:')
        sr = {}

        # Frequency/OOD labels (needed for combined model evaluation)
        freq = torch.zeros(n_ent, dtype=torch.long)
        freq.scatter_add_(0, train_t[:, 0], torch.ones(len(train_t), dtype=torch.long))
        freq.scatter_add_(0, train_t[:, 2], torch.ones(len(train_t), dtype=torch.long))
        tau = int(np.percentile(freq[freq > 0].numpy(), 25))
        h_test, r_test, t_test = test_t[:, 0], test_t[:, 1], test_t[:, 2]
        min_freq = torch.minimum(freq[h_test], freq[t_test])
        is_emerging = min_freq <= tau

        # Energy baseline
        print(f'    Energy...')
        set_seed(seed)
        m = EnergyBasedKGE(n_ent, n_rel).to(device)
        m = train_model(m, train_t)
        sr['energy_base'] = evaluate_ood(m, train_t, test_t)
        # Energy + Coverage (post-hoc combination with batch normalization)
        combined_unc = combine_with_coverage(m, test_t)
        cov_h = m.coverage[h_test.to(device), r_test.to(device)].cpu()
        cov_t = m.coverage[t_test.to(device), r_test.to(device)].cpu()
        is_novel = (~is_emerging) & ((cov_h == 0) | (cov_t == 0))
        is_ood = (is_emerging | is_novel).numpy().astype(int)
        if is_ood.sum() > 0 and (is_ood == 0).sum() > 0:
            sr['energy_plus_cov'] = {
                'overall': float(roc_auc_score(is_ood, combined_unc)),
                'overall_aupr': float(average_precision_score(is_ood, combined_unc)),
            }
        else:
            sr['energy_plus_cov'] = {'overall': float('nan'), 'overall_aupr': float('nan')}
        # FPR@95TPR for combined model
        ood_scores = combined_unc[is_ood == 1]
        id_scores = combined_unc[is_ood == 0]
        if len(ood_scores) > 0 and len(id_scores) > 0:
            tpr95_thresh = np.percentile(ood_scores, 5)
            sr['energy_plus_cov']['fpr95'] = float((id_scores >= tpr95_thresh).mean())
        print(f'      Energy: {sr["energy_base"]["overall"]:.4f}, +Cov: {sr["energy_plus_cov"]["overall"]:.4f}')
        del m; torch.cuda.empty_cache(); gc.collect()

        # MCDropout baseline
        print(f'    MCDropout...')
        set_seed(seed)
        m = MCDropoutKGE(n_ent, n_rel).to(device)
        m = train_model(m, train_t)
        sr['mcdropout_base'] = evaluate_ood(m, train_t, test_t)
        combined_unc = combine_with_coverage(m, test_t)
        if is_ood.sum() > 0 and (is_ood == 0).sum() > 0:
            sr['mcdropout_plus_cov'] = {
                'overall': float(roc_auc_score(is_ood, combined_unc)),
                'overall_aupr': float(average_precision_score(is_ood, combined_unc)),
            }
        else:
            sr['mcdropout_plus_cov'] = {'overall': float('nan'), 'overall_aupr': float('nan')}
        ood_scores = combined_unc[is_ood == 1]
        id_scores = combined_unc[is_ood == 0]
        if len(ood_scores) > 0 and len(id_scores) > 0:
            tpr95_thresh = np.percentile(ood_scores, 5)
            sr['mcdropout_plus_cov']['fpr95'] = float((id_scores >= tpr95_thresh).mean())
        print(f'      MCDropout: {sr["mcdropout_base"]["overall"]:.4f}, +Cov: {sr["mcdropout_plus_cov"]["overall"]:.4f}')
        del m; torch.cuda.empty_cache(); gc.collect()

        # CoverageOnly
        print(f'    CoverageOnly...')
        set_seed(seed)
        m = CoverageOnly(n_ent, n_rel).to(device)
        m = train_model(m, train_t)
        sr['coverage_only'] = evaluate_ood(m, train_t, test_t)
        print(f'      CoverageOnly: {sr["coverage_only"]["overall"]:.4f}')
        del m; torch.cuda.empty_cache(); gc.collect()

        # CAGP (learned combination)
        print(f'    CAGP...')
        set_seed(seed)
        m = CAGP(n_ent, n_rel).to(device)
        m = train_model(m, train_t, unc_weight=0.1, is_gp=True)
        sr['cagp'] = evaluate_ood(m, train_t, test_t)
        alpha_val = torch.sigmoid(m.alpha_logit).item()
        sr['cagp']['alpha'] = alpha_val
        print(f'      CAGP: {sr["cagp"]["overall"]:.4f}, alpha={alpha_val:.3f}')
        del m; torch.cuda.empty_cache(); gc.collect()

        seed_results.append(sr)

    # Aggregate — include AUPR, FPR@95, alpha
    summary = {}
    for method in seed_results[0]:
        for metric in ['overall', 'overall_aupr', 'fpr95', 'emerging', 'novel', 'alpha']:
            vals = [s[method].get(metric, float('nan')) for s in seed_results]
            vals = [v for v in vals if isinstance(v, (int, float)) and not np.isnan(v)]
            if vals:
                summary[f'{method}_{metric}'] = {'mean': float(np.mean(vals)), 'std': float(np.std(vals))}
    exp2_results[ds_name] = {'per_seed': seed_results, 'summary': summary}

exp2_results['elapsed_seconds'] = time.time() - t0
with open('outputs/exp2_baseline_coverage.json', 'w') as f:
    json.dump(exp2_results, f, indent=2)

print(f'\n\nExp 2 done in {exp2_results["elapsed_seconds"]:.0f}s')
for ds in DATASETS:
    print(f'\n{ds}:')
    for k, v in sorted(exp2_results[ds]['summary'].items()):
        print(f'  {k}: {v["mean"]:.4f} ± {v["std"]:.4f}')
    s = exp2_results[ds]['summary']
    print(f'\n  KEY: Post-hoc +Cov vs learned CAGP:')
    for base in ['energy', 'mcdropout']:
        phoc = s.get(f'{base}_plus_cov_overall', {}).get('mean', 0)
        cagp = s.get('cagp_overall', {}).get('mean', 0)
        print(f'    {base}+Cov={phoc:.4f} vs CAGP={cagp:.4f} (gap={cagp-phoc:+.4f})')


    # UKGE baseline
    print('\n' + '='*50)
    print('UKGE Baseline')
    print('='*50)
    m = UKGE(n_ent, n_rel)
    m.to(device)
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    m.precompute_coverage(train_t)
    
    for epoch in range(EPOCHS):
        losses = []
        for h, r, t in make_batches(train_t, BATCH_SIZE):
            opt.zero_grad()
            score = m(h, r, t)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(score, torch.ones_like(score))
            loss.backward()
            opt.step()
            losses.append(loss.item())
        if epoch % 10 == 0:
            print(f'  Epoch {epoch}: Loss={np.mean(losses):.4f}')
    
    m.eval()
    with torch.no_grad():
        r = evaluate_ood(m, train_t, test_t)
    results['UKGE'] = r
    print(f'  AUROC: {r["overall"]:.4f}')
    del m; torch.cuda.empty_cache(); gc.collect()

    # SNGP baseline
    print('\n' + '='*50)
    print('SNGP Baseline')
    print('='*50)
    m = SNGPModel(n_ent, n_rel)
    m.to(device)
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    m.precompute_coverage(train_t)
    
    for epoch in range(EPOCHS):
        losses = []
        for h, r, t in make_batches(train_t, BATCH_SIZE):
            opt.zero_grad()
            score = m(h, r, t)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(score, torch.ones_like(score))
            loss.backward()
            opt.step()
            losses.append(loss.item())
        if epoch % 10 == 0:
            print(f'  Epoch {epoch}: Loss={np.mean(losses):.4f}')
    
    m.eval()
    with torch.no_grad():
        r = evaluate_ood(m, train_t, test_t)
    results['SNGP'] = r
    print(f'  AUROC: {r["overall"]:.4f}')
    del m; torch.cuda.empty_cache(); gc.collect()

    # DeepEnsemble baseline
    print('\n' + '='*50)
    print('DeepEnsemble Baseline')
    print('='*50)
    m = DeepEnsembleKGE(n_ent, n_rel, n_members=3)
    m.to(device)
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    m.precompute_coverage(train_t)
    
    for epoch in range(EPOCHS):
        losses = []
        for h, r, t in make_batches(train_t, BATCH_SIZE):
            opt.zero_grad()
            score = m(h, r, t)
            loss = torch.nn.functional.binary_cross_entropy_with_logits(score, torch.ones_like(score))
            loss.backward()
            opt.step()
            losses.append(loss.item())
        if epoch % 10 == 0:
            print(f'  Epoch {epoch}: Loss={np.mean(losses):.4f}')
    
    m.eval()
    with torch.no_grad():
        r = evaluate_ood(m, train_t, test_t)
    results['DeepEnsemble'] = r
    print(f'  AUROC: {r["overall"]:.4f}')
    del m; torch.cuda.empty_cache(); gc.collect()

In [ ]:
# Experiment 2.5: Reparameterization Sampling Ablationprint('='*70)print('EXPERIMENT 2.5: REPARAMETERIZATION ABLATION')print('Paper claim: Without reparam, AUROC drops 0.92→0.72 (WN18RR), 0.90→0.68 (YAGO)')print('='*70)reparam_results = {}for ds_name in ['WN18RR', 'YAGO3-10']:    if ds_name not in DATASETS:        continue    data, n_ent, n_rel = DATASETS[ds_name]    train_t = torch.tensor(data['train'], device=device)    test_t = torch.tensor(data['test'], device=device)    seeds = DATASET_SEEDS.get(ds_name, SEEDS)[:3]  # Use 3 seeds for ablation        with_reparam = []    without_reparam = []        for seed in seeds:        print(f'\n  {ds_name} seed {seed}:')        # With reparameterization (standard CAGP)        set_seed(seed)        m = CAGP(n_ent, n_rel)        m.to(device)        m = train_model(m, train_t, unc_weight=0.1, is_gp=True)        r = evaluate_ood(m, train_t, test_t)        with_reparam.append(r['overall'])        print(f'    With reparam: {r["overall"]:.4f}')        del m; torch.cuda.empty_cache(); gc.collect()                # Without reparameterization         set_seed(seed)        m = CAGPNoReparam(n_ent, n_rel)        m.to(device)        m = train_model(m, train_t, unc_weight=0.1, is_gp=True)        r = evaluate_ood(m, train_t, test_t)        without_reparam.append(r['overall'])        print(f'    Without reparam: {r["overall"]:.4f}')        del m; torch.cuda.empty_cache(); gc.collect()        reparam_results[ds_name] = {        'with_reparam': {'mean': float(np.mean(with_reparam)), 'std': float(np.std(with_reparam))},        'without_reparam': {'mean': float(np.mean(without_reparam)), 'std': float(np.std(without_reparam))},        'delta_pp': float(np.mean(with_reparam) - np.mean(without_reparam)) * 100,    }    print(f'  {ds_name}: {np.mean(with_reparam):.3f} → {np.mean(without_reparam):.3f} '          f'(Δ={reparam_results[ds_name]["delta_pp"]:.1f}pp)')with open('outputs/exp2_5_reparam_ablation.json', 'w') as f:    json.dump(reparam_results, f, indent=2)print('\nReparameterization ablation complete.')

if gd_data is not None:
    ---
    ## Experiment 3: GDELT Pipeline

    New temporal KG. Addresses "only ICEWS14 is non-circular" critique.
    Runs CAGP, CoverageOnly, GPOnly, Energy on GDELT.
else:
    print('GDELT data not available — skipping Experiment 3')

In [ ]:
if gd_data is not None:
    # Cell 9: Experiment 3 — GDELT temporal KG (with strict-split for non-circular evaluation)
    print('='*70)
    print('EXPERIMENT 3: GDELT TEMPORAL KG (30 epochs, 3 seeds)')
    print('='*70)
    
    # Verify temporal non-circularity
    print('\n--- Temporal Split Verification ---')
    if gd_timestamps is not None and 'train' in gd_timestamps and 'test' in gd_timestamps:
        tr_ts = gd_timestamps['train'].numpy()
        te_ts = gd_timestamps['test'].numpy()
        print(f'  Train timestamps: min={tr_ts.min()}, max={tr_ts.max()}, unique={len(np.unique(tr_ts))}')
        print(f'  Test timestamps:  min={te_ts.min()}, max={te_ts.max()}, unique={len(np.unique(te_ts))}')
        temporal_gap = te_ts.min() - tr_ts.max()
        print(f'  Temporal gap (test_min - train_max): {temporal_gap}')
        overlap_pct = (te_ts <= tr_ts.max()).mean() * 100
        print(f'  Test triples with timestamp <= train max: {overlap_pct:.1f}%')
        if overlap_pct < 50:
            print('  ✓ CONFIRMED: Chronological split — novel-context detection is NON-CIRCULAR')
        else:
            print('  ⚠ WARNING: High temporal overlap — results may still have circularity concerns')
    else:
        print('  ⚠ No timestamps available — treating as simulated split')
    
    exp3_results = {}
    t0 = time.time()
    
    gd_train, gd_test = gd_data['train'], gd_data['test']
    print(f'\nGDELT: {gd_nent} entities, {gd_nrel} relations, {len(gd_train)} train, {len(gd_test)} test')
    
    models_to_run = [
        ('CAGP', lambda: CAGP(gd_nent, gd_nrel), True, 0.1),
        ('CoverageOnly', lambda: CoverageOnly(gd_nent, gd_nrel), False, 0.0),
        ('GPOnly', lambda: GPOnly(gd_nent, gd_nrel), True, 0.0),
        ('Energy', lambda: EnergyBasedKGE(gd_nent, gd_nrel), False, 0.0),
    ]
    
    seed_results = []
    for seed in DATASET_SEEDS.get('GDELT', SEEDS_3):
        print(f'\n--- Seed {seed} ---')
        sr = {}
        for name, model_fn, is_gp, uw in models_to_run:
            print(f'  {name}...')
            set_seed(seed)
            m = model_fn()
            m = train_model(m, gd_train, is_gp=is_gp, unc_weight=uw)
    
            # Standard evaluation
            r = evaluate_ood(m, gd_train, gd_test)
            sr[name] = r
            print(f'    standard: AUROC={r["overall"]:.4f}, AUPR={r.get("overall_aupr","N/A")}, '
                  f'emerging={r.get("emerging","N/A")}, novel={r.get("novel","N/A")}')
    
            # Strict-split evaluation (removes inverse-relation leakage)
            r_strict = evaluate_strict_split(m, gd_train, gd_test, gd_nrel)
            sr[f'{name}_strict'] = r_strict
            print(f'    strict:   AUROC={r_strict.get("overall","N/A")}, removed={r_strict["removed_pct"]:.1f}%')
    
            del m; torch.cuda.empty_cache(); gc.collect()
        seed_results.append(sr)
    
    # Aggregate
    summary = {}
    all_keys = list(seed_results[0].keys())
    for method in all_keys:
        for metric in ['overall', 'overall_aupr', 'fpr95', 'emerging', 'novel']:
            vals = [s[method].get(metric, float('nan')) for s in seed_results]
            vals = [v for v in vals if isinstance(v, (int, float)) and not np.isnan(v)]
            if vals:
                summary[f'{method}_{metric}'] = {'mean': float(np.mean(vals)), 'std': float(np.std(vals))}
    
    for method in ['CAGP_strict', 'CoverageOnly_strict', 'GPOnly_strict', 'Energy_strict']:
        rpcts = [s[method].get('removed_pct', float('nan')) for s in seed_results]
        rpcts = [v for v in rpcts if not np.isnan(v)]
        if rpcts:
            summary[f'{method}_removed_pct'] = {'mean': float(np.mean(rpcts)), 'std': float(np.std(rpcts))}
    
    exp3_results = {
        'dataset': 'GDELT',
        'temporal_benchmark': True,
        'num_entities': gd_nent,
        'num_relations': gd_nrel,
        'timestamp_stats': {
            'train_min': int(gd_timestamps['train'].min()) if gd_timestamps else None,
            'train_max': int(gd_timestamps['train'].max()) if gd_timestamps else None,
            'test_min': int(gd_timestamps['test'].min()) if gd_timestamps else None,
            'test_max': int(gd_timestamps['test'].max()) if gd_timestamps else None,
        },
        'per_seed': seed_results,
        'summary': summary,
        'elapsed_seconds': time.time() - t0,
    }
    
    with open('outputs/exp3_gdelt.json', 'w') as f:
        json.dump(exp3_results, f, indent=2)
    
    print(f'\n\nExp 3 done in {exp3_results["elapsed_seconds"]:.0f}s')
    
    print('\n' + '='*70)
    print('GDELT — KEY NUMBERS FOR PAPER (NON-CIRCULAR TEMPORAL BENCHMARK)')
    print('='*70)
    
    print('\n1. OVERALL OOD DETECTION:')
    for name in ['CAGP', 'CoverageOnly', 'GPOnly', 'Energy']:
        auroc = summary.get(f'{name}_overall', {})
        aupr = summary.get(f'{name}_overall_aupr', {})
        fpr = summary.get(f'{name}_fpr95', {})
        print(f'  {name:15s}: AUROC={auroc.get("mean",0):.4f}±{auroc.get("std",0):.4f}  '
              f'AUPR={aupr.get("mean",0):.4f}  FPR@95={fpr.get("mean",0):.4f}')
    
    print('\n2. EMERGING ENTITY AUROC:')
    for name in ['CAGP', 'CoverageOnly', 'GPOnly', 'Energy']:
        em = summary.get(f'{name}_emerging', {})
        print(f'  {name:15s}: {em.get("mean","N/A")}')
    
    print('\n3. NOVEL-CONTEXT AUROC (non-circular on temporal KG):')
    for name in ['CAGP', 'CoverageOnly', 'GPOnly', 'Energy']:
        nv = summary.get(f'{name}_novel', {})
        print(f'  {name:15s}: {nv.get("mean","N/A")}')
    
    print('\n4. STRICT-SPLIT (removes inverse-relation leakage):')
    for name in ['CAGP', 'CoverageOnly', 'GPOnly', 'Energy']:
        auroc = summary.get(f'{name}_strict_overall', {})
        removed = summary.get(f'{name}_strict_removed_pct', {})
        print(f'  {name:15s}: AUROC={auroc.get("mean","N/A")}  (removed {removed.get("mean",0):.1f}%)')
    
    print('\n5. KEY QUESTION: Does CAGP > CoverageOnly on emerging entities?')
    cagp_em = summary.get('CAGP_emerging', {}).get('mean', 0)
    cov_em = summary.get('CoverageOnly_emerging', {}).get('mean', 0)
    gp_em = summary.get('GPOnly_emerging', {}).get('mean', 0)
    delta = cagp_em - cov_em
    print(f'   CAGP emerging = {cagp_em:.4f}, CoverageOnly emerging = {cov_em:.4f}')
    print(f'   Delta = {delta*100:+.1f}pp')
    if delta > 0.02:
        print('   ✓ YES: Semantic component provides meaningful lift → complementarity confirmed on non-circular benchmark')
    elif delta > 0:
        print('   ~ Marginal: Small lift, consistent with ICEWS14/18 pattern')
    else:
        print('   ✗ NO: Coverage alone is sufficient (same as ICEWS pattern)')
    
    
else:
    print('GDELT data not available — skipping Experiment 3')

---
## Experiment 4: R-GCN / CompGCN

Tests whether GNN encoders still produce relation-agnostic variance (confirming Theorem 1).
Also tests GNN+Coverage augmentation vs CAGP.

In [ ]:
# Cell 10: Experiment 4 — R-GCN / CompGCN
print('='*70)
print('EXPERIMENT 4: R-GCN / CompGCN (30 epochs, 3 seeds)')
print('='*70)

exp4_results = {}
t0 = time.time()

gnn_configs = [
    ('R-GCN', 'rgcn', False),
    ('R-GCN+Cov', 'rgcn', True),
    ('CompGCN', 'compgcn', False),
    ('CompGCN+Cov', 'compgcn', True),
]

for ds_name, (data, n_ent, n_rel) in DATASETS.items():
    print(f'\n{"="*60}')
    print(f'Dataset: {ds_name} ({n_ent} entities, {n_rel} relations)')
    print(f'{"="*60}')
    train_t, test_t = data['train'], data['test']
    ds_results = {}

    for seed in DATASET_SEEDS.get(ds_name, SEEDS_3):
        print(f'\n  Seed {seed}:')
        seed_r = {}
        for cfg_name, enc_type, use_cov in gnn_configs:
            print(f'    {cfg_name}...')
            set_seed(seed)
            try:
                m = GNNModel(n_ent, n_rel, DIM, enc_type, use_cov)
                m = train_gnn_model(m, train_t)
                r = evaluate_ood(m, train_t, test_t)
                seed_r[cfg_name] = r
                print(f'      overall={r["overall"]:.4f}, novel={r.get("novel","N/A")}')
            except Exception as e:
                print(f'      ERROR: {e}')
                seed_r[cfg_name] = {'overall': float('nan'), 'error': str(e)}
            del m; torch.cuda.empty_cache(); gc.collect()

        # Also run CAGP for comparison (full model with margin loss)
        print(f'    CAGP (reference)...')
        set_seed(seed)
        m = CAGP(n_ent, n_rel)
        m = train_model(m, train_t, unc_weight=0.1, is_gp=True)
        seed_r['CAGP'] = evaluate_ood(m, train_t, test_t)
        alpha_val = torch.sigmoid(m.alpha_logit).item()
        seed_r['CAGP']['alpha'] = alpha_val
        print(f'      overall={seed_r["CAGP"]["overall"]:.4f}, alpha={alpha_val:.3f}')
        del m; torch.cuda.empty_cache(); gc.collect()

        ds_results[f'seed_{seed}'] = seed_r

    # Aggregate — include AUPR, FPR@95, alpha
    summary = {}
    all_methods = [c[0] for c in gnn_configs] + ['CAGP']
    for method in all_methods:
        for metric in ['overall', 'overall_aupr', 'fpr95', 'emerging', 'novel', 'alpha']:
            vals = []
            for seed in DATASET_SEEDS.get(ds_name, SEEDS_3):
                v = ds_results.get(f'seed_{seed}', {}).get(method, {}).get(metric, float('nan'))
                if isinstance(v, (int, float)) and not np.isnan(v):
                    vals.append(v)
            if vals:
                summary[f'{method}_{metric}'] = {'mean': float(np.mean(vals)), 'std': float(np.std(vals))}

    exp4_results[ds_name] = {'per_seed': ds_results, 'summary': summary}

exp4_results['elapsed_seconds'] = time.time() - t0
with open('outputs/exp4_rgcn_compgcn.json', 'w') as f:
    json.dump(exp4_results, f, indent=2)

print(f'\n\nExp 4 done in {exp4_results["elapsed_seconds"]:.0f}s')

# === KEY RESULTS FOR REVIEWER ===
for ds in DATASETS:
    s = exp4_results[ds]['summary']
    print(f'\n{"="*70}')
    print(f'{ds} — KEY NUMBERS FOR PAPER')
    print(f'{"="*70}')

    print('\n  THEOREM 1 VALIDATION (GNN novel-context AUROC should be ~0.5):')
    print('  (GNN energy uncertainty is relation-agnostic → random for novel-context)')
    for gnn in ['R-GCN', 'CompGCN']:
        nv = s.get(f'{gnn}_novel', {})
        nv_cov = s.get(f'{gnn}+Cov_novel', {})
        print(f'    {gnn:12s} novel AUROC = {nv.get("mean","N/A")}  (should be ~0.5)')
        print(f'    {gnn}+Cov novel AUROC = {nv_cov.get("mean","N/A")}  (coverage fixes it)')

    print(f'\n  OVERALL COMPARISON:')
    for method in ['R-GCN', 'R-GCN+Cov', 'CompGCN', 'CompGCN+Cov', 'CAGP']:
        ov = s.get(f'{method}_overall', {})
        print(f'    {method:12s}: AUROC={ov.get("mean",0):.4f}±{ov.get("std",0):.4f}')
    print(f'    → CAGP should beat all GNN variants (learned α > fixed combination)')

---
## Experiment 5: 10-Seed ICEWS14 (Statistical Power)

Tests whether CAGP vs U_str difference is statistically significant with 10 seeds.
Downloads ICEWS14 temporal KG and runs all methods with seeds [42, 123, 456, 789, 1011, 1213, 1415, 1617, 1819, 2021].
Reports paired t-test p-values.


# Experiment 5B: ICEWS18
print('='*70)
print('EXPERIMENT 5B: ICEWS18 (Temporal OOD — Extended Benchmark)')
print('Paper claim: 5-seed validation on extended temporal corpus')
print('='*70)

ic18_results = {}
for seed in DATASET_SEEDS.get('ICEWS18', SEEDS_5):
    print(f'\nSeed {seed}:')
    set_seed(seed)
    
    train_t = torch.tensor(ic18_data['train'], device=device)
    test_t = torch.tensor(ic18_data['test'], device=device)
    
    results = {}
    
    # CAGP
    print('  CAGP...')
    m = CAGP(ic18_nent, ic18_nrel)
    m.to(device)
    m = train_model(m, train_t, unc_weight=0.1, is_gp=True)
    r = evaluate_ood(m, train_t, test_t)
    results['CAGP'] = r
    print(f'    AUROC: {r["overall"]:.4f}')
    del m; torch.cuda.empty_cache(); gc.collect()
    
    # GP-KGE (no coverage)
    print('  GP-KGE...')
    m = GPKGE(ic18_nent, ic18_nrel)
    m.to(device)
    m = train_model(m, train_t, unc_weight=0.1, is_gp=True)
    r = evaluate_ood(m, train_t, test_t)
    results['GP-KGE'] = r
    print(f'    AUROC: {r["overall"]:.4f}')
    del m; torch.cuda.empty_cache(); gc.collect()
    
    # RelCondVar
    print('  RelCondVar...')
    m = RelCondVar(ic18_nent, ic18_nrel)
    m.to(device)
    m = train_model(m, train_t, unc_weight=0.1, is_gp=False)
    r = evaluate_ood(m, train_t, test_t)
    results['RelCondVar'] = r
    print(f'    AUROC: {r["overall"]:.4f}')
    del m; torch.cuda.empty_cache(); gc.collect()
    
    ic18_results[seed] = results

with open('outputs/exp5b_icews18.json', 'w') as f:
    json.dump(ic18_results, f, indent=2)
print('\nICEWS18 evaluation complete.')

In [ ]:
# Experiment 5: 10-Seed ICEWS14
print('='*70)
print('EXPERIMENT 5: 10-SEED ICEWS14 (Statistical Power)')
print('='*70)

SEEDS_10 = [42, 123, 456, 789, 1011, 1213, 1415, 1617, 1819, 2021]

# Download ICEWS14
print('Downloading ICEWS14...')
icews_dir = 'data/ICEWS14'
os.makedirs(icews_dir, exist_ok=True)
icews_base = 'https://raw.githubusercontent.com/Liyyy2122/TiRGN/main/data/ICEWS14'
for split in ['train', 'valid', 'test']:
    path = f'{icews_dir}/{split}.txt'
    if not os.path.exists(path):
        try:
            print(f'  Downloading ICEWS14/{split}.txt...')
            urllib.request.urlretrieve(f'{icews_base}/{split}.txt', path)
        except urllib.error.HTTPError as e:
            print(f'  Skipping {split}.txt ({e}) — not required for OOD evaluation')

# Load with timestamps
ic_data, ic_timestamps, ic_nent, ic_nrel = load_triples_temporal(icews_dir)
print(f'  ICEWS14: {ic_nent} entities, {ic_nrel} relations')
print(f'  Train: {len(ic_data["train"])} triples, Test: {len(ic_data["test"])} triples')

# Verify temporal split
if ic_timestamps:
    tr_ts = ic_timestamps['train'].numpy()
    te_ts = ic_timestamps['test'].numpy()
    print(f'  Train timestamps: [{tr_ts.min()}, {tr_ts.max()}]')
    print(f'  Test timestamps:  [{te_ts.min()}, {te_ts.max()}]')
    overlap = (te_ts <= tr_ts.max()).mean() * 100
    print(f'  Temporal overlap: {overlap:.1f}%')

ic_train, ic_test = ic_data['train'], ic_data['test']

models_to_run = [
    ('CAGP', lambda: CAGP(ic_nent, ic_nrel), True, 0.1),
    ('CoverageOnly', lambda: CoverageOnly(ic_nent, ic_nrel), False, 0.0),
    ('GPOnly', lambda: GPOnly(ic_nent, ic_nrel), True, 0.0),
    ('Energy', lambda: EnergyBasedKGE(ic_nent, ic_nrel), False, 0.0),
]

t0 = time.time()
seed_results = []
for seed in SEEDS_10:
    print(f'\n--- Seed {seed} ---')
    sr = {}
    for name, model_fn, is_gp, uw in models_to_run:
        print(f'  {name}...')
        set_seed(seed)
        m = model_fn()
        m = train_model(m, ic_train, is_gp=is_gp, unc_weight=uw)
        r = evaluate_ood(m, ic_train, ic_test)
        sr[name] = r
        print(f'    AUROC={r["overall"]:.4f}, emerging={r.get("emerging","N/A")}, novel={r.get("novel","N/A")}')

        # Strict split
        r_strict = evaluate_strict_split(m, ic_train, ic_test, ic_nrel)
        sr[f'{name}_strict'] = r_strict
        print(f'    strict: AUROC={r_strict.get("overall","N/A")}, removed={r_strict["removed_pct"]:.1f}%')

        del m; torch.cuda.empty_cache(); gc.collect()
    seed_results.append(sr)

# Aggregate with 10 seeds
summary = {}
for method in ['CAGP', 'CoverageOnly', 'GPOnly', 'Energy']:
    for metric in ['overall', 'overall_aupr', 'fpr95', 'emerging', 'novel']:
        vals = [s[method].get(metric, float('nan')) for s in seed_results]
        vals = [v for v in vals if isinstance(v, (int, float)) and not np.isnan(v)]
        if vals:
            summary[f'{method}_{metric}'] = {
                'mean': float(np.mean(vals)),
                'std': float(np.std(vals)),
                'values': vals,
            }
    # Strict split
    for metric in ['overall']:
        vals = [s[f'{method}_strict'].get(metric, float('nan')) for s in seed_results]
        vals = [v for v in vals if isinstance(v, (int, float)) and not np.isnan(v)]
        if vals:
            summary[f'{method}_strict_{metric}'] = {
                'mean': float(np.mean(vals)),
                'std': float(np.std(vals)),
                'values': vals,
            }

# Statistical tests: CAGP vs CoverageOnly
from scipy import stats
print('\n' + '='*70)
print('ICEWS14 10-SEED RESULTS')
print('='*70)

print('\n1. Overall AUROC (10-seed mean ± std):')
for name in ['CAGP', 'CoverageOnly', 'GPOnly', 'Energy']:
    d = summary.get(f'{name}_overall', {})
    print(f'  {name:15s}: {d.get("mean",0):.4f} ± {d.get("std",0):.4f}')

print('\n2. Emerging Entity AUROC:')
for name in ['CAGP', 'CoverageOnly', 'GPOnly', 'Energy']:
    d = summary.get(f'{name}_emerging', {})
    print(f'  {name:15s}: {d.get("mean",0):.4f} ± {d.get("std",0):.4f}')

print('\n3. Strict-Split AUROC:')
for name in ['CAGP', 'CoverageOnly', 'GPOnly', 'Energy']:
    d = summary.get(f'{name}_strict_overall', {})
    print(f'  {name:15s}: {d.get("mean",0):.4f} ± {d.get("std",0):.4f}')

print('\n4. Statistical Significance (paired t-test, CAGP vs CoverageOnly):')
for metric in ['overall', 'emerging', 'novel']:
    cagp_vals = summary.get(f'CAGP_{metric}', {}).get('values', [])
    cov_vals = summary.get(f'CoverageOnly_{metric}', {}).get('values', [])
    if len(cagp_vals) == len(cov_vals) and len(cagp_vals) >= 2:
        t_stat, p_val = stats.ttest_rel(cagp_vals, cov_vals)
        sig = '***' if p_val < 0.001 else '**' if p_val < 0.01 else '*' if p_val < 0.05 else 'n.s.'
        delta = np.mean(cagp_vals) - np.mean(cov_vals)
        print(f'  {metric:12s}: delta={delta:+.4f}, t={t_stat:.3f}, p={p_val:.4f} {sig}')

exp5_results = {
    'dataset': 'ICEWS14',
    'num_seeds': len(SEEDS_10),
    'seeds': SEEDS_10,
    'per_seed': seed_results,
    'summary': {k: {'mean': v['mean'], 'std': v['std']} for k, v in summary.items()},
    'elapsed_seconds': time.time() - t0,
}

with open('outputs/exp5_icews14_10seed.json', 'w') as f:
    json.dump(exp5_results, f, indent=2, default=str)

print(f'\nExp 5 done in {exp5_results["elapsed_seconds"]:.0f}s')



# Experiment 5B: ICEWS18
print('='*70)
print('EXPERIMENT 5B: ICEWS18 (Temporal OOD — Extended Benchmark)')
print('Paper claim: 5-seed validation on extended temporal corpus')
print('='*70)

ic18_results = {}
for seed in DATASET_SEEDS.get('ICEWS18', SEEDS_5):
    print(f'\nSeed {seed}:')
    set_seed(seed)
    
    train_t = torch.tensor(ic18_data['train'], device=device)
    test_t = torch.tensor(ic18_data['test'], device=device)
    
    results = {}
    
    # CAGP
    print('  CAGP...')
    m = CAGP(ic18_nent, ic18_nrel)
    m.to(device)
    m = train_model(m, train_t, unc_weight=0.1, is_gp=True)
    r = evaluate_ood(m, train_t, test_t)
    results['CAGP'] = r
    print(f'    AUROC: {r["overall"]:.4f}')
    del m; torch.cuda.empty_cache(); gc.collect()
    
    # GP-KGE (no coverage)
    print('  GP-KGE...')
    m = GPKGE(ic18_nent, ic18_nrel)
    m.to(device)
    m = train_model(m, train_t, unc_weight=0.1, is_gp=True)
    r = evaluate_ood(m, train_t, test_t)
    results['GP-KGE'] = r
    print(f'    AUROC: {r["overall"]:.4f}')
    del m; torch.cuda.empty_cache(); gc.collect()
    
    # RelCondVar
    print('  RelCondVar...')
    m = RelCondVar(ic18_nent, ic18_nrel)
    m.to(device)
    m = train_model(m, train_t, unc_weight=0.1, is_gp=False)
    r = evaluate_ood(m, train_t, test_t)
    results['RelCondVar'] = r
    print(f'    AUROC: {r["overall"]:.4f}')
    del m; torch.cuda.empty_cache(); gc.collect()
    
    ic18_results[seed] = results

with open('outputs/exp5b_icews18.json', 'w') as f:
    json.dump(ic18_results, f, indent=2)
print('\nICEWS18 evaluation complete.')

## Figure: Two Types of OOD in Knowledge Graphs

Generates Figure 1 for the paper: illustrates why entity variance fails on novel relational contexts.


In [ ]:
# Conceptual Figure: Two OOD Types in Knowledge Graphs
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Color scheme
COLOR_ID = '#4CAF50'       # Green for in-distribution
COLOR_EMERGE = '#FF5722'   # Red-orange for emerging
COLOR_NOVEL = '#2196F3'    # Blue for novel context
COLOR_BG = '#FAFAFA'
COLOR_LIGHT = '#E8E8E8'

# === Panel 1: Emerging Entity ===
ax = axes[0]
ax.set_xlim(0, 10); ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.set_facecolor(COLOR_BG)
ax.set_title('(a) Emerging Entity', fontsize=13, fontweight='bold', pad=12)

# Known entities (filled, large)
for (x, y, label) in [(3, 7, 'A'), (7, 7, 'B'), (5, 4, 'C'), (3, 2, 'D')]:
    circle = plt.Circle((x, y), 0.6, color=COLOR_ID, alpha=0.8, zorder=3)
    ax.add_patch(circle)
    ax.text(x, y, label, ha='center', va='center', fontsize=11, fontweight='bold', color='white', zorder=4)

# Emerging entity (dashed, small)
circle = plt.Circle((8, 3), 0.6, color=COLOR_EMERGE, alpha=0.3, zorder=3, linestyle='--', linewidth=2, fill=True)
ax.add_patch(circle)
circle_border = plt.Circle((8, 3), 0.6, color=COLOR_EMERGE, fill=False, linewidth=2, linestyle='--', zorder=4)
ax.add_patch(circle_border)
ax.text(8, 3, 'E', ha='center', va='center', fontsize=11, fontweight='bold', color=COLOR_EMERGE, zorder=5)
ax.text(8, 1.8, 'NEW\n(1 triple)', ha='center', va='center', fontsize=8, color=COLOR_EMERGE, style='italic')

# Relations (edges)
for (x1, y1, x2, y2, r) in [(3,7,7,7,'r1'), (7,7,5,4,'r2'), (5,4,3,2,'r3'), (3,7,5,4,'r1'), (5,4,8,3,'r2')]:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.2, connectionstyle='arc3,rad=0.1'))
    mx, my = (x1+x2)/2, (y1+y2)/2
    ax.text(mx+0.2, my+0.3, r, fontsize=7, color='gray', style='italic')

# Variance indicator
ax.text(8, 9.2, 'σ² = HIGH', fontsize=10, ha='center', color=COLOR_EMERGE, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.3', facecolor=COLOR_EMERGE, alpha=0.15))
ax.text(5, 9.2, 'σ² = low', fontsize=10, ha='center', color=COLOR_ID,
        bbox=dict(boxstyle='round,pad=0.3', facecolor=COLOR_ID, alpha=0.15))

ax.set_xticks([]); ax.set_yticks([])
for spine in ax.spines.values(): spine.set_visible(False)

# === Panel 2: Novel Context ===
ax = axes[1]
ax.set_xlim(0, 10); ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.set_facecolor(COLOR_BG)
ax.set_title('(b) Novel Relational Context', fontsize=13, fontweight='bold', pad=12)

# All entities are known (filled, large)
for (x, y, label) in [(3, 7, 'A'), (7, 7, 'B'), (5, 4, 'C'), (3, 2, 'D'), (8, 3, 'E')]:
    circle = plt.Circle((x, y), 0.6, color=COLOR_ID, alpha=0.8, zorder=3)
    ax.add_patch(circle)
    ax.text(x, y, label, ha='center', va='center', fontsize=11, fontweight='bold', color='white', zorder=4)

# Known relations (gray)
for (x1, y1, x2, y2, r) in [(3,7,7,7,'r1'), (7,7,5,4,'r2'), (5,4,3,2,'r3'), (3,7,5,4,'r1')]:
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color='gray', lw=1.2, connectionstyle='arc3,rad=0.1'))
    mx, my = (x1+x2)/2, (y1+y2)/2
    ax.text(mx+0.2, my+0.3, r, fontsize=7, color='gray', style='italic')

# Novel relation (blue, thick, dashed)
ax.annotate('', xy=(8, 3), xytext=(3, 7),
            arrowprops=dict(arrowstyle='->', color=COLOR_NOVEL, lw=2.5, linestyle='--',
                          connectionstyle='arc3,rad=-0.15'))
ax.text(6.2, 5.8, 'r4', fontsize=10, color=COLOR_NOVEL, fontweight='bold', style='italic')
ax.text(6.2, 5.1, 'NEW relation\nfor entity A', fontsize=7, color=COLOR_NOVEL, ha='center', style='italic')

# Variance indicator — SAME as ID!
ax.text(3, 9.2, 'σ²(A) = low', fontsize=10, ha='center', color=COLOR_ID,
        bbox=dict(boxstyle='round,pad=0.3', facecolor=COLOR_ID, alpha=0.15))
ax.text(8, 9.2, 'σ²(E) = low', fontsize=10, ha='center', color=COLOR_ID,
        bbox=dict(boxstyle='round,pad=0.3', facecolor=COLOR_ID, alpha=0.15))
ax.text(5.5, 0.5, 'Entity variance CANNOT detect this!\nσ² is relation-agnostic (Theorem 1)',
        fontsize=8, ha='center', color=COLOR_NOVEL, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor=COLOR_NOVEL, alpha=0.1))

ax.set_xticks([]); ax.set_yticks([])
for spine in ax.spines.values(): spine.set_visible(False)

# === Panel 3: Detection Summary ===
ax = axes[2]
ax.set_xlim(0, 10); ax.set_ylim(0, 10)
ax.set_aspect('equal')
ax.set_facecolor(COLOR_BG)
ax.set_title('(c) Complementary Detection\n(static benchmarks; weaker on temporal)', fontsize=11, fontweight='bold', pad=12)

# Table-like layout
headers = ['', 'Emerging\nEntity', 'Novel\nContext']
row_labels = ['U_sem\n(variance)', 'U_str\n(coverage)', 'CAGP\n(combined)']
cell_colors = [
    [COLOR_EMERGE, COLOR_LIGHT],  # U_sem: good on emerging, bad on novel
    [COLOR_LIGHT, COLOR_NOVEL],   # U_str: bad on emerging, good on novel
    [COLOR_EMERGE, COLOR_NOVEL],  # CAGP: good on both
]
cell_text = [
    ['✓ HIGH', '✗ LOW'],
    ['~ PARTIAL', '✓ PERFECT'],
    ['✓ HIGH', '✓ PERFECT'],
]

for j, header in enumerate(headers):
    x = 1.5 + j * 3
    ax.text(x, 9, header, ha='center', va='center', fontsize=9, fontweight='bold')

for i, (label, colors, texts) in enumerate(zip(row_labels, cell_colors, cell_text)):
    y = 7 - i * 2.5
    ax.text(1.5, y, label, ha='center', va='center', fontsize=9, fontweight='bold')
    for j, (color, text) in enumerate(zip(colors, texts)):
        x = 4.5 + j * 3
        rect = FancyBboxPatch((x-1.2, y-0.7), 2.4, 1.4, boxstyle='round,pad=0.1',
                               facecolor=color, alpha=0.3, edgecolor=color, linewidth=1.5)
        ax.add_patch(rect)
        ax.text(x, y, text, ha='center', va='center', fontsize=9, fontweight='bold',
                color='#333333')

# Separator lines
ax.axhline(y=8, xmin=0.1, xmax=0.9, color='gray', linewidth=0.5)

ax.set_xticks([]); ax.set_yticks([])
for spine in ax.spines.values(): spine.set_visible(False)

plt.tight_layout(pad=2)
plt.savefig('outputs/conceptual_figure.pdf', dpi=300, bbox_inches='tight')
plt.savefig('outputs/conceptual_figure.png', dpi=300, bbox_inches='tight')
print('Saved: outputs/conceptual_figure.pdf and outputs/conceptual_figure.png')
plt.show()



---
## Summary & Download

In [ ]:
# Cell 11: Final Summary
print('='*80)
print('ALL EXPERIMENTS COMPLETE')
print('='*80)

total_time = (
    exp1_results.get('elapsed_seconds', 0) +
    exp2_results.get('elapsed_seconds', 0) +
    globals().get('exp3_results', {}).get('elapsed_seconds', 0) +
    exp4_results.get('elapsed_seconds', 0) +
    exp5_results.get('elapsed_seconds', 0)
)
print(f'Total time: {total_time:.0f}s ({total_time/60:.1f} min)\n')

# Consolidated summary
consolidated = {
    'exp1_margin_loss': exp1_results,
    'exp2_baseline_coverage': exp2_results,
    'exp3_gdelt': globals().get('exp3_results', {}),
    'exp4_rgcn_compgcn': exp4_results,
    'exp5_icews14_10seed': exp5_results,
    'total_seconds': total_time,
}
with open('outputs/all_neurips_experiments.json', 'w') as f:
    json.dump(consolidated, f, indent=2, default=str)

print('Output files:')
for f in sorted(os.listdir('outputs')):
    if f.endswith('.json') or f.endswith('.pdf') or f.endswith('.png'):
        size = os.path.getsize(f'outputs/{f}') / 1024
        print(f'  outputs/{f} ({size:.1f} KB)')

print('\nKey results to check:')
print('  1. Margin loss: w_unc=0.1 vs 0.0 difference should be small (<1pp)')
print('  2. Baseline+Cov: Energy+Cov and MCDropout+Cov should NOT match CAGP')
print('  3. GDELT: CAGP should beat single components on temporal KG (non-circular!)')
print('  4. R-GCN/CompGCN: GNN energy should be relation-agnostic → low AUROC')
print('  5. ICEWS14 10-seed: CAGP vs CoverageOnly with statistical significance')
print('  6. Conceptual figure: outputs/conceptual_figure.pdf')



In [ ]:
# Cell 12: Download results
# In Colab, use this to download the consolidated JSON:
try:
    from google.colab import files
    files.download('outputs/all_neurips_experiments.json')
    print('Download started!')
except ImportError:
    print('Not in Colab. Results are in outputs/ directory.')